# Generate a window-labeling artifact

Builds an SME-facing review page for one equipment's flagged (Yellow/Red) test windows:
each window's raw sensor values, plus its full ranked sensor-attribution scores, bundled
small enough to publish as a Claude Artifact.

**Prerequisites** (run once per equipment, outside this notebook):
1. `python main.py --data ... --output output_<EQ> --config configs/<EQ>_data.yaml --event-labels ...`
2. `python scripts/score_full_timeline.py --data ... --model-dir output_<EQ>/artefacts --split-ids ... --split test --output output_<EQ>/evaluation/window_scores.parquet`

This notebook only reads the outputs of those two steps — it does not train or score anything itself.

**What this notebook produces**: a `build/<equipment_id>/` folder containing `index.html` and a
`data/` folder of chunked raw-window-data files. It does **not** publish anything — a notebook has
no access to the Artifact tool. The last cell prints the exact instruction to hand to Claude Code
to publish the folder it just built.

## Parameters — edit these for the equipment you're reviewing

In [ ]:
from pathlib import Path

REPO = Path("/Users/mes-ass-0196-mac-anurav/anurav/Autoencoder")

EQUIPMENT_ID = "5K512B"
DATA = REPO / f"data/{EQUIPMENT_ID}/{EQUIPMENT_ID}_combined_with_events_clean.parquet"
WINDOW_SCORES = REPO / f"output_{EQUIPMENT_ID}/evaluation/window_scores.parquet"
MODEL_DIR = REPO / f"output_{EQUIPMENT_ID}/artefacts"

ZONES = "yellow,red"          # which zones to pull into the review set
MAX_WINDOWS = 150             # capped by score (highest first) to stay under the
                              # Artifact tool's 64MB-per-publish limit -- see the
                              # sizing check at the end of Step 2 before raising this
WINDOWS_PER_CHUNK = 15        # ~5MB per chunk file at ~150 sensors x 120 rows

BUILD_DIR = REPO / "notebooks" / "labeling_build" / EQUIPMENT_ID
TEMPLATE_PATH = REPO / "notebooks" / "labeling_template.html"

## Step 1 — export raw window data + fresh sensor attribution

Delegates to `scripts/export_labeling_windows.py`, which re-runs the trained model's own
inference on just the flagged windows to recover the *full* per-sensor ranking (not the
truncated top-k persisted in `window_scores.parquet`), and verifies two ways that every
exported window is exactly the one that was actually scored: its raw-row timestamps must
match `window_scores.parquet`'s recorded `window_start`/`window_end`, and its recomputed
anomaly score must match the stored one. Either check failing aborts the export rather than
risk silently mismatched data reaching the review page.

In [ ]:
import subprocess, sys

RAW_EXPORT_DIR = BUILD_DIR / "raw_export"  # intermediate only -- not published

subprocess.run(
    [
        sys.executable, str(REPO / "scripts/export_labeling_windows.py"),
        "--data", str(DATA),
        "--window-scores", str(WINDOW_SCORES),
        "--model-dir", str(MODEL_DIR),
        "--output-dir", str(RAW_EXPORT_DIR),
        "--equipment-id", EQUIPMENT_ID,
        "--zones", ZONES,
    ],
    cwd=REPO, check=True,
)

## Step 2 — bundle into size-capped chunks + a manifest

The manifest carries per-window metadata and the *full* ranked sensor-attribution list (small,
always loaded up front so the sidebar and sensor search work immediately). Raw per-sensor time
series are the bulk of the data and are split across a handful of chunk files, fetched lazily
by the page only when a window is actually opened.

In [ ]:
import json, math

index = json.loads((RAW_EXPORT_DIR / f"{EQUIPMENT_ID}_index.json").read_text())
selected = index["windows"][:MAX_WINDOWS]  # already sorted by anomaly_score descending
print(f"Selected {len(selected)} / {index['n_windows']} exported windows (top by anomaly_score).")

data_dir = BUILD_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)
manifest_windows = []
n_chunks = math.ceil(len(selected) / WINDOWS_PER_CHUNK)

for chunk_idx in range(n_chunks):
    chunk_entries = selected[chunk_idx * WINDOWS_PER_CHUNK : (chunk_idx + 1) * WINDOWS_PER_CHUNK]
    chunk_payload = {}
    for entry in chunk_entries:
        record = json.loads((RAW_EXPORT_DIR / entry["file"]).read_text())
        chunk_payload[str(entry["window_id"])] = {
            "timestamps": record["timestamps"],
            "raw_values": record["raw_values"],
        }
        manifest_windows.append({
            "id": entry["window_id"],
            "start": record["window_start"],
            "end": record["window_end"],
            "zone": record["zone"],
            "score": record["anomaly_score"],
            "chunk": f"data/chunk_{chunk_idx}.json",
            "sensors": [
                {
                    "name": name,
                    "error": record["sensor_error"][name],
                    "ratio": record["sensor_error_ratio"].get(name),
                    "pct": record["sensor_contribution_pct"][name],
                }
                for name in record["ranked_sensors"]
            ],
        })
    chunk_path = data_dir / f"chunk_{chunk_idx}.json"
    chunk_path.write_text(json.dumps(chunk_payload))
    print(f"chunk_{chunk_idx}.json: {len(chunk_entries)} windows, {chunk_path.stat().st_size/1e6:.2f} MB")

manifest = {
    "equipment_id": EQUIPMENT_ID,
    "sensor_columns": index["sensor_columns"],
    "thresholds": index["thresholds"],
    "windows": manifest_windows,
}

In [ ]:
# Sizing check -- keep this comfortably under the Artifact tool's 64MB-per-publish cap.
manifest_json = json.dumps(manifest)
chunk_total = sum(f.stat().st_size for f in data_dir.glob("*.json"))
print(f"manifest: {len(manifest_json)/1e6:.2f} MB (embedded inline in index.html)")
print(f"chunks:   {chunk_total/1e6:.2f} MB across {n_chunks} file(s)")
print(f"TOTAL:    {(len(manifest_json) + chunk_total)/1e6:.2f} MB -- if this exceeds ~60MB, lower MAX_WINDOWS and re-run from Step 2")

## Step 3 — assemble the final page

Injects the manifest into `labeling_template.html` (the review UI: a searchable window list, a
stacked per-sensor time-series chart defaulting to each window's top-5 attributed sensors, a
searchable sensor picker for any of the equipment's sensors, and Green/Yellow/Red override
buttons that persist to the artifact's `db` capability once published).

In [ ]:
template = TEMPLATE_PATH.read_text()
html = template.replace("__MANIFEST_JSON__", manifest_json)
html = html.replace("__EQUIPMENT_ID__", EQUIPMENT_ID)

out_path = BUILD_DIR / "index.html"
out_path.write_text(html)
print(f"Wrote {out_path} ({out_path.stat().st_size/1e6:.2f} MB)")

## Done — publishing this is a Claude Code step, not a notebook step

This notebook has no access to the Artifact tool, so it stops here, with a ready-to-publish
folder on disk. Hand the next cell's printed instruction to Claude Code to actually publish it
(new equipment → a new artifact; re-running for one already published → ask Claude to update
that same artifact's URL instead of creating a new one).

In [ ]:
chunk_files = sorted(p.name for p in data_dir.glob("*.json"))
print(f"""
Ask Claude Code to publish this as an Artifact:

  file_path: {out_path.relative_to(REPO)}
  root:      {BUILD_DIR.relative_to(REPO)}
  files:     {{ "data/{{name}}": "data/{{name}}" for name in chunk_files }}
  capabilities: {{"db": {{}}}}
""")